In [10]:
import pandas as pd
from pathlib import Path

In [ ]:
DATA_DIR = Path.cwd().parent / "data"
SUBMISSION_CLIENT = Path.cwd().parent / "submission" / "client.csv"
SUBMISSION_CAMPAIGN = Path.cwd().parent / "submission" / "campaign.csv"
SUBMISSION_ECONOMICS = Path.cwd().parent / "submission" / "economics.csv"

In [13]:
archivos_gz = list(DATA_DIR.glob("*.csv.gz"))   # glob busca todos los archivos al interior de DATA_DIR que cumplan con el patrón *.csv.gz. Se queda con una lista que tiene la ruta completa desde la raíz hasta cada archivo en data
datos_marketing = [pd.read_csv(archivos) for archivos in archivos_gz]   # Se crea una lista de dataframes que corresponden a los datos de cada archivo contenido en achivos_gz
datos_consolidados = pd.concat(datos_marketing, ignore_index=True)   # Se concatenan todos los dataframes de la lista datos_marketing en un solo dataframe llamado datos_consolidado. ignore_index=True asegura que los índices se reindexen de manera continua.

datos_consolidados

,Unnamed: 0,client_id,age,job,marital,education,credit_default,mortgage,month,day,contact_duration,number_contacts,previous_campaign_contacts,previous_outcome,cons_price_idx,euribor_three_months,campaign_outcome
0,0,0,56,housemaid,married,basic.4y,no,no,may,13,261,1,0,nonexistent,93.994,4.857,no
1,1,1,57,services,married,high.school,unknown,no,may,19,149,1,0,nonexistent,93.994,4.857,no
2,2,2,37,services,married,high.school,no,yes,may,23,226,1,0,nonexistent,93.994,4.857,no
3,3,3,40,admin.,married,basic.6y,no,no,may,27,151,1,0,nonexistent,93.994,4.857,no
4,4,4,56,services,married,high.school,no,no,may,3,307,1,0,nonexistent,93.994,4.857,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,4113,41183,73,retired,married,professional.course,no,yes,nov,30,334,1,0,nonexistent,94.767,1.028,yes
41184,4114,41184,46,blue-collar,married,professional.course,no,no,nov,6,383,1,0,nonexistent,94.767,1.028,no
41185,4115,41185,56,retired,married,university.degree,no,yes,nov,24,189,2,0,nonexistent,94.767,1.028,no
41186,4116,41186,44,technician,married,professional.course,no,no,nov,17,442,1,0,nonexistent,94.767,1.028,yes


In [ ]:
#------------------------
# PRIMER ARCHVIO 
# -----------------------

COLUMNAS_CLIENT_CSV = ["client_id", "age", "job", "marital",        # Se construye una lista con las columnas requeridas en el orden indicado
                       "education", "credit_default", "mortgage"]


def limpiar_job(series):                                 # Se construye la función auxiliar limpiar_job 
    series = series.str.replace(".", "", regex=False)    # Se eliminan los puntos de la columna job
    series = series.str.replace("-", "_", regex=False)   # Se reemplazan los guiones por guiones bajos en la columna job
    return series

def limpiar_education(series):
    series = series.str.replace(".", "_", regex=False)    # Se reemplazan los puntos por guiones bajos en la columna education
    series = series.mask(series == "unknown")             # Se usa mask para que en todas aquellas oportunidades en que haya una observación con unknown se tenga un NaN como salida
    return series

def limpiar_credit_default_y_mortgage(series):            # Se construye la función auxiliar limpiar_credit_default_y_mortgage
    return (series == "yes").astype(int)                  # Si series == "yes" astype(int) devuelve 1, de lo contrario devuelve 0


client = datos_consolidados[COLUMNAS_CLIENT_CSV].copy()   # Se crea un dataframe client que contiene las columnas indicadas en COLUMNAS_CLIENT_CSV sobre el cual realizar el procesamiento ya construido en las funciones
client["job"] = limpiar_job(client["job"])                # Se aplica la función limpiar_job a la columna job del dataframe client
client["education"] = limpiar_education(client["education"])
client["credit_default"] = limpiar_credit_default_y_mortgage(client["credit_default"])
client["mortgage"] = limpiar_credit_default_y_mortgage(client["mortgage"])

In [ ]:
# -----------------------
# SEGUNDO ARCHIVO 
# -----------------------

COLUMNAS_CAMPAIGN_CSV = ["client_id", "number_contacts", "contact_duration", "previous_campaign_contacts", 
                         "previous_outcome","campaign_outcome", "last_contact_date"]


MESES = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
         "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12}


def limpiar_previous_outcome(series):                 # Se construye la función auxiliar limpiar_previous_outcome
    return (series == "success").astype(int)


def limpiar_campaign_outcome(series):                 # Se construye la función auxiliar limpiar_campaign_outcome
    return (series == "yes").astype(int)


def construir_last_contact_date(datos_consolidados, año=2022):      # Se construye la función auxiliar construir_last_contact_date
    fecha = pd.to_datetime(pd.DataFrame({"year": año,
                                         "month": datos_consolidados["month"].map(MESES),
                                         "day": datos_consolidados["day"]}))
    return fecha.dt.strftime("%Y-%m-%d")

datos_consolidados["last_contact_date"] = construir_last_contact_date(datos_consolidados)   # Se aplica la función construir_last_contact_date a los datos consolidados para crear la columna last_contact_date


campaign = datos_consolidados[COLUMNAS_CAMPAIGN_CSV].copy()   # Se crea un dataframe campaign que contiene las columnas indicadas en COLUMNAS_CAMPAIGN_CSV sobre el cual realizar el procesamiento ya construido en las funciones
campaign["previous_outcome"] = limpiar_previous_outcome(campaign["previous_outcome"])
campaign["campaign_outcome"] = limpiar_campaign_outcome(campaign["campaign_outcome"])


In [ ]:
# -----------------------
# TERCER ARCHIVO
# -----------------------

COLUMNAS_ECONOMICS_CSV = ["client_id", "cons_price_idx", "euribor_three_months"]  # Se construye una lista con las columnas requeridas en el orden indicado

economcis = datos_consolidados[COLUMNAS_ECONOMICS_CSV].copy()   # Se crea un dataframe economcis que contiene las columnas indicadas en COLUMNAS_ECONOMICS_CSV sobre el cual realizar el procesamiento ya construido en las funciones

In [ ]:
# Guardado de las tablas
client.to_csv(SUBMISSION_CLIENT, sep=",",index=False)
campaign.to_csv(SUBMISSION_CAMPAIGN, sep=",", index=False)
economcis.to_csv(SUBMISSION_ECONOMICS, sep=",", index=False)